# Vision Quest — FINAL Colab Runner (clean, full pipeline)
Early Prediction of Batting Shot-Type from Cricket Videos (IE 643).

This is the single clean notebook that reproduces everything end to end:
dataset → frames → high-res pose → train video → train pose → results → demo.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save.
Run the cells top to bottom. Nothing here depends on the Kaggle API.

### 1. Check GPU (expect a Tesla T4 line)

In [ ]:
!nvidia-smi

### 2. Clone repo + install deps
Put YOUR GitHub repo URL below. The repo must already contain the latest code
(high-res pose extraction + the fixes).

In [ ]:
REPO_URL = 'https://github.com/<YOUR_USERNAME>/VisionQuest_IE643_CricketShotPrediction.git'
!git clone $REPO_URL
%cd VisionQuest_IE643_CricketShotPrediction
!pip install -q -r requirements.txt
print('\nSetup done.')

### 3. Get the dataset (via Google Drive — reliable)
Upload your cricket-shots zip to Google Drive once, then set its path below.
(If you prefer a direct upload instead, replace this cell's body with:
`from google.colab import files; up=files.upload()` then unzip `list(up)[0]`.)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('data/raw', exist_ok=True)
# EDIT this to your zip's path in Drive:
ZIP = '/content/drive/MyDrive/cricket-shots-ipl-2023.zip'
!unzip -q -o "$ZIP" -d data/raw/
print('raw contents:', os.listdir('data/raw'))

### 4. Inspect classes (confirm they match configs/default.yaml)
Expected 7: drive, pull, cut, sweep, flick, slog, misc.

In [ ]:
!python -m src.data.download --inspect --raw data/raw

### 5. Build split + cache 112px frames (for the video branch)

In [ ]:
!python -m src.data.extract_frames --config configs/default.yaml --set paths.raw_dataset=data/raw

### 6. Extract pose (HIGH-RES, two-stage crop)  — SLOW (~60–75 min)
Decodes clips at 256px, crops to the batter, re-detects on the zoom. Watch the
final line: **Mean per-frame detection** should be ~0.55. Downloads a model on
first run.

In [ ]:
!python -m src.data.extract_pose --config configs/default.yaml

### 7. Train the VIDEO branch (Setting A). ~30–40 min. val top1 should reach ~0.5–0.57.

In [ ]:
!python -m src.train.train_video --config configs/default.yaml

### 8. Train the POSE branch (Setting B). ~2 min. 60 epochs, patience 20.

In [ ]:
!python -m src.train.train_pose --config configs/default.yaml --set train.epochs=60 train.early_stop_patience=20

### 9. Temporal analysis + figures (Subtask 4) — writes results + the 3 plots

In [ ]:
!python -m src.eval.temporal_analysis --config configs/default.yaml
import json
r = json.load(open('results/temporal_results.json'))
def pct(x): return f'{x*100:.1f}'
print('VIDEO Top-1 @ 25/50/75/100:', [pct(r['video_acc'][s]['top1']) for s in ['0.25','0.5','0.75','1.0']])
print('POSE  Top-1 @ 25/50/75/100:', [pct(r['pose_acc'][s]['top1']) for s in ['0.25','0.5','0.75','1.0']])
print('video macro-F1 (full):', round(r.get('video_full_macro_f1',0),3))

### 10. Show the headline figure

In [ ]:
from IPython.display import Image
Image('results/figures/accuracy_vs_ratio.png')

### 11. Download results + checkpoints (numbers, figures, trained models)

In [ ]:
!zip -r vq_outputs.zip results checkpoints
from google.colab import files; files.download('vq_outputs.zip')

### 12. Launch the demo interface (for the DEMO video)
Prints a public gradio.live link. Open it, upload a clip, move the observation
slider. Screen-record this (both faces on cam) for the demo video.

In [ ]:
!python -m app.gradio_app --config configs/default.yaml --share